# 1. Example API usage

So you want to uplaod, retrieve or delete data for the HGTD Production Database automatically with the help of a python script, not needing to open and use the frontend with tar files.

This example shows how to authenticate, get or post data for an endpoint that concerns a single attribute of a Module: in this example `thickness_U (mm)`.

Later in this notebook, we also prepare and upload a whole set of electrical measurements, right from here in the python API client. This can be streamlined to be a new script for FADAPro or Alvin!

In [1]:
import api

## 1.1 Authentication

There are two parts, which currently are fully independent:

- making sure the user logs in via CERN SSO, and
- getting an access token, which expires after a while but will be active for making the actual request to the ProdDB

### 1.1.1 User name

We have a function that requests a small check against the CERN `/openid-connect` endpoint which allows us to make sure a user has to "log in" before the following steps. Obviously, I'll not put my password here so you'll see an error, as expected.

In [ ]:
api.get_user('your-username','your-password','6-digits-for-2FA')

If you used the correct combination of username, password and TOTP code for 2FA, you should receive

`('your-username', '200, OK')`

We want to record the username in upcoming requests to the ProdDB, to associate a user with an upload for easy tracking and making sure we know whom to contact in case of questions.

Because of that, we save the user name as `auth_user` (and a `responseText`) for later.

In [ ]:
auth_user, last_responseText = api.get_user('your-username','your-password','6-digits-for-2FA')

So imagine, you (or the user via command line prompt) did the previous step and you received a valid user name. For now, let's use mine `anstein` as if I had done this handshake with the CERN SSO.

In [2]:
auth_user = 'anstein'

### 1.1.2 Access token for DB

The database is protected, i.e. the endpoints require for example a method that only proceeds with requests when an access token is sent along with the request.

In the `api.py` file, you'll see some hard-coded elements for `applicationDetails`, these are items that should not be shared outside of the realm of ATLAS / HGTD. These are static over time, while each access token that you request has some expiry time.

In [ ]:
api.get_access_token()

As you see, an access token will be returned. We'll save a token for later. Each time you call this function, you get a new one, the old one is not "recycled", although technically it should be possible.

In [3]:
access_token = api.get_access_token()

## 1.2 Communication with a suitable ProdDB endpoint

There is a webpage that shows all implemented endpoints: https://hgtddb-api.web.cern.ch/schema/swagger-ui/

In our case, we want to add a value for some attribute for a given part. Let's say you know the serial number and you have found the part in the database, either programmatically (which I don't show in this example just yet), or by inspecting the parts via the frontend.

Let's say we pick this example module: https://nginx-hgtddb.app.cern.ch/viewparts/6573

The URL gives away what is called `part_id`, a unique identifier in the database that is not the serial number, but just an internal counter. We need this `part_id` for many operations with the DB, because this is what the endpoints usually accept. In our case, it's the 6573.

We now have a look at implemented endpoints, and how they are used by the existing infrastructure (views in the frontend). We will find that `https://hgtddb-api.web.cern.ch/hgtddb/partattrpost` is the right endpoint to add a value for a single attribute for a certain `part_id` and that the endpoint just wants a dictionary / JSON-like input. See how it is done for the frontend here: https://gitlab.cern.ch/hgtd-database/hgtd/-/blob/master/frontend_gui/src/views/addpartstoattributes.vue?ref_type=heads#L330-338

To tell the database which attribute should be affected by our upcomming operation, we need to know its `attribute_id`. This concept is very similar to the `part_id`. There are as many ways to find this id as there are ways to (programmatically) find `part_id`s, let's say you know it because you have access to an admin page https://nginx-hgtddb.app.cern.ch/attributelist and find that the parameter is internally stored with id 1077.

### 1.2.1 Build the data in the suitable structure

In this basic example, one value for one attribute for one part, the structure can be built as follows:

In [4]:
attributes_for_my_module = {
    'is_record_deleted': 'F', # we want to record this as something that is not "deleted", so we put "F"
    'value': 1.7, # the value we want to insert for this attribute, here I just picked an example
    'part': 6573, # the part_id we talked about
    'attribute': 1077, # the attribute_id we talked about
    'record_insertion_user': auth_user # to store who did this upload in the database
}

### 1.2.2 Build the final request to the ProdDB

With this structure as what is called a `payload`, we can use the `api` module with its capabilities to post such a standard request. Remember the endpoint that was discussed needs to be used as a parameter for this request. The module is written such that you only need the actual `/endpoint` part, not the full URL.

In [5]:
endpoint_to_post = '/partattrpost'

We set `dryrun = True` to only test this function call before actually doing it.

In [6]:
last_responseText = api.post_information(endpoint_to_post, attributes_for_my_module, dryrun = True)

>>> Dryrun post operation with endpoint /partattrpost
>>> and payload {'is_record_deleted': 'F', 'value': 1.7, 'part': 6573, 'attribute': 1077, 'record_insertion_user': 'anstein'}


We actually perform this operation when we set `dryrun = False`.

In [7]:
last_responseText = api.post_information(endpoint_to_post, attributes_for_my_module, dryrun = False)

If there are no errors showing up, let's inspect the `last_responseText`:

In [8]:
last_responseText

'200, OK'

Congratulations, you have just used the pure-python API to write new information to our HGTD Production Database! Response codes in the 200s are good, 201 is for a successful upload and the text mentions that this new node was created with what you gave as payload. 200 would be an indicator that the data already existed, just your request was an update and successfully processed as well (the latter happens if you re-run this notebook).

## 1.3 Inspect the result

Check with https://nginx-hgtddb.app.cern.ch/viewparts/6573 to see the attribute was correctly uplaoded to the DB!

And we have another tool to inspect the part further, we can get the full list of its attributes.

In [9]:
endpoint_to_get_attributes = '/partattrlist/6573/'

In [10]:
attributes_of_part, responseText = api.fetch_information(endpoint_to_get_attributes)

In [11]:
attributes_of_part

[{'attr_list_record_id': 1035391,
  'is_record_deleted': 'F',
  'record_insertion_time': '2025-08-27T12:44:38Z',
  'record_insertion_user': 'anstein',
  'record_del_flag_time': None,
  'record_del_flag_user': '',
  'value': '1.7',
  'record_lastupdate_user': 'ATLAS_HGTD_PROD',
  'record_lastupdate_time': '2026-05-23T08:26:43Z',
  'part': {'part_id': 6573,
   'is_record_deleted': 'F',
   'record_insertion_time': '2024-11-29T10:13:48Z',
   'record_insertion_user': 'anstein',
   'barcode': 'none5',
   'serial_number': 'none5',
   'version': '1',
   'name_label': 'Dummy Mod',
   'installed_date': None,
   'removed_date': None,
   'installed_by_user': '',
   'removed_by_user': '',
   'comment_description': 'Dummy Mod to test relations',
   'record_lastupdate_time': '2025-05-03T14:02:38Z',
   'record_lastupdate_user': 'anstein',
   'production_date': None,
   'batch_number': '',
   'kind_of_part': {'kind_of_part_id': 1005,
    'display_name': 'Module',
    'is_record_deleted': 'F',
    'reco

This is quite detailed, the important thing is that the `value` was recorded for the `attribute`, while you also see the details for those definitions and the part itself. This also holds the `serial_number` and `part_id`, and there are similar endpoints which hold such info not for one module, but all parts of a certain Kind of Part (KoP), for example.

## 1.4 Extending the example

You have now seen how a dictionary gets uploaded carrying some value per attribute. We need to understand the usage of the more involved endpoints (bulk upload, multiple parts, or multiple attributes per part or a combination of both, and of course the `.csv`- or `.tar`-file upload for e.g. electrical measurements).

### 1.4.1 Communication with a suitable ProdDB endpoint for tar file upload

Part: e.g. https://nginx-hgtddb.app.cern.ch/viewparts/28832

Endpoint: see https://gitlab.cern.ch/hgtd-database/hgtd/-/blob/master/frontend_gui/src/views/addmeasurementthresholdbulk.vue?ref_type=heads#L289-304 we need the endpoint `/module_threshold_measurements_many` with `multipart/form-data` content type

```
    const partslist = {
                data_file: this.file_upload,
                location: this.extractNumbersFromLastBrackets(this.location),
                //manufacturer: this.extractNumbersFromLastBrackets(this.manufacturer_id),
                run_type: this.run_type,
                is_record_deleted: 'F',
                run_begin_timestamp: this.start_date,
                run_end_timestamp: this.end_date,
                comment_description: this.comm_desc,

            };
            console.log(JSON.parse(JSON.stringify(partslist)));
            //alert(JSON.parse(JSON.stringify(partslist)));
            axios.post(`${this.apiUrl}module_threshold_measurements_many`, partslist, { headers: { 'Content-Type': 'multipart/form-data' } }

            )
```

Data format: prepare a tar archive here on the fly, DB can not understand individual files for bulk upload, the alternative would be singular csv files (but then they have no metadata); according to example in here https://gitlab.cern.ch/hgtd-database/hgtd/-/blob/master/frontend_gui/src/views/addmeasurementthresholdbulk.vue?ref_type=heads#L86 we want accept=".tar,.tgz,.tar.gz"

Hint: the `multipart/form-data` is added automatically to headers with correct length, when `files` parameter is set in the request.

#### 1.4.1.1 Build the data in the suitable structure

In this tar file example, the electrical measurements need to be combined into one archive, even though we have all individual files available here.

For this, the `tarfile` package is used, where we generate / use the tar-archive for the upload with the POST request.

location: https://nginx-hgtddb.app.cern.ch/locationlist

run_type: we need to figure out which of these we want to allow, this should include `<step>_<site>_<something else>`

The file to upload is to be created from the existing measurement / analysis results, and in my case I'd like to test this with a cern reception test file.

We have to prepare a tar that has a main directory, and below those, the different scans, within those the configuration and at the next level the actual data. Example:

```
main_dir_name
├── bump_connection
│   └── module_0
│       └── bump_connection_plot.pdf
├── chargeScan
│   └── B_None_On_all_Inj_row_N_100_Vth_380_Q_12
│       ├── analysis_results.csv
│       ├── DB_upload_info.yaml
│       ├── meas_parameters.yaml
│       ├── metadata.txt
│       └── module_0
│           ├── DB_metadata.yaml
│           ├── DB_results.csv
│           ├── occupancy_map_injected.pdf
│           ├── occupancy_map.pdf
│           ├── SN.txt
│           ├── threshold_map_fit_A.pdf
│           ├── threshold_map_fit_mu.pdf
│           ├── threshold_map_fit_noise.pdf
│           ├── threshold_map.pdf
│           ├── threshold_scan.pdf
│           ├── toa_mean_map_all.pdf
│           ├── toa_mean_map_inj.pdf
│           ├── toa_mean_vs_Q.pdf
│           ├── toa_std_map_all.pdf
│           ├── toa_std_map_inj.pdf
│           ├── toa_std_vs_Q.pdf
│           ├── tot_mean_map_all.pdf
│           ├── tot_mean_map_inj.pdf
│           ├── tot_mean_vs_Q.pdf
│           ├── tot_std_map_all.pdf
│           ├── tot_std_map_inj.pdf
│           └── tot_std_vs_Q.pdf
├── delayScan
│   └── B_None_On_all_Inj_row_N_100_Vth_1023_Q_12
│       ├── DB_upload_info.yaml
│       └── module_0
│           ├── counts.pdf
│           ├── DB_metadata.yaml
│           ├── DB_results.csv
│           ├── LSB.pdf
│           ├── SN.txt
│           ├── toa_mean.pdf
│           └── toa_std.pdf
├── digitalScan
│   └── B_None_On_all_Inj_row_N_100_Vth_1023_Q_12
│       ├── DB_upload_info.yaml
│       └── module_0
│           ├── DB_metadata.yaml
│           ├── DB_results.csv
│           ├── digital_scan.pdf
│           └── SN.txt
├── last_vthc
│   └── asic0_vthc.txt
├── thresScan
│   ├── B_None_On_all_Inj_row_N_100_Vth_380_Q_12
│   │   ├── analysis_results.csv
│   │   ├── DB_upload_info.yaml
│   │   ├── meas_parameters.yaml
│   │   ├── metadata.txt
│   │   └── module_0
│   │       ├── DB_metadata.yaml
│   │       ├── DB_results.csv
│   │       ├── SN.txt
│   │       ├── threshold_map.pdf
│   │       └── threshold_scan.pdf
│   └── B_None_On_all_Inj_row_N_100_Vth_380_Q_36
│       ├── analysis_results.csv
│       ├── DB_upload_info.yaml
│       ├── meas_parameters.yaml
│       ├── metadata.txt
│       └── module_0
│           ├── DB_metadata.yaml
│           ├── DB_results.csv
│           ├── SN.txt
│           ├── threshold_map.pdf
│           └── threshold_scan.pdf
├── vthcScan
│   └── B_None_On_all_Inj_row_N_100_Vth_380_Q_12
│       ├── analysis_results.csv
│       ├── asic0_vthc.txt
│       ├── DB_upload_info.yaml
│       ├── meas_parameters.yaml
│       ├── metadata.txt
│       └── module_0
│           ├── DB_metadata.yaml
│           ├── DB_results.csv
│           ├── SN.txt
│           ├── threshold_map.pdf
│           └── threshold_scan.pdf
└── widthScan
    └── B_None_On_all_Inj_row_N_100_Vth_1023_Q_12
        ├── DB_upload_info.yaml
        └── module_0
            ├── counts.pdf
            ├── DB_metadata.yaml
            ├── DB_results.csv
            ├── LSB.pdf
            ├── SN.txt
            ├── tot_mean.pdf
            └── tot_std.pdf


```

Now imagine you have a folder with the analysis results. We can use the python `tarfile` package to build a GNU-formatted `tar` archive with the correct pattern.

In [12]:
import tarfile

In [16]:
# from the location of this script / notebook: a path to an analysis directory with the suitable structure
path_to_analysis_dir = 'another_test/'

# name of the tar (choose it yourself)
target_tar = 'something.tar'

In [17]:
# build the tar in the correct format, using the main directory still as the first level within that tar
# we include the `format` option to make sure that even with macOS, the content will be as if it had been prepared on linux with GNU tar
with tarfile.open(target_tar, "w", format=tarfile.GNU_FORMAT) as tar:
    tar.add(path_to_analysis_dir)

The next step differs a bit from the standard one-by-one attribute upload from Section 1.2, we have to prepare two parts: the file (binary) and the accompanying human-readable JSON-like data that holds some standard information about this measurement.

In [18]:
# first item is the file itself
files_payload = {'data_file': ('something.tar', open('something.tar','rb'))}

In [19]:
files_payload

{'data_file': ('something.tar', <_io.BufferedReader name='something.tar'>)}

In [20]:
# second we need the corresponding data with all extra fields
data_payload = {'is_record_deleted': 'F', # we want to record this as something that is not "deleted", so we put "F"
                'location': "1521", # measurement site / location (could be your institute location, cern, clean room etc.) - we need to hardcode somewhere or retrieve the list of locatoins to choose from, because again this needs an ID
                'run_type': "test_upload_via_API_2", # measurement type, e.g. the full name of the step (assembly_<insititute>, loading_<after some TCs> etc.)
                'run_begin_timestamp': "2025-09-02", # begin of measurement data, usual time and date format
                'run_end_timestamp': "2025-09-02", # end of measurement data, usual time and date format
                'comment_description': "python_API_client", # comment
                'record_insertion_user': "anstein"} # to store who did this upload in the database

In [21]:
endpoint_module_bulk_meas = '/module_threshold_measurements_many'

#### 1.4.1.2 Build the final request to the ProdDB

In [22]:
last_responseText = api.post_information(endpoint_module_bulk_meas, data_payload, dryrun = True, content_type = 'multipart/form-data', files_payload = files_payload)

>>> Dryrun post operation with endpoint /module_threshold_measurements_many
>>> and payload {'is_record_deleted': 'F', 'location': '1521', 'run_type': 'test_upload_via_API_2', 'run_begin_timestamp': '2025-09-02', 'run_end_timestamp': '2025-09-02', 'comment_description': 'python_API_client', 'record_insertion_user': 'anstein'}


In [23]:
last_responseText = api.post_information(endpoint_module_bulk_meas, data_payload, dryrun = False, content_type = 'multipart/form-data', files_payload = files_payload)

In [24]:
last_responseText

'201, Created'

### 1.4.2 Inspecting the result

Actually, I uploaded this example above multiple times to demonstrate what happens in that case: we get a new "Run Number" for each upload. That is how to differentiate between different entries with the same `data_payload` (i.e. run type).

Check with https://monitoring-hgtddb.app.cern.ch/monitoring/module_threshold?format=json&serial_number=testModule_4_Annika or on the main view for module measuremetns at https://nginx-hgtddb.app.cern.ch/modulethresholdlist to see what we just uploaded! (If you prepared your own tar for another Serial Number, just exchange the name in the query to inspect your very own measurements).

## 1.5 Ideas for improvement

- We can opt to delete the tar file, or create it in some temporary directory
- Check first for what is already available (either as a separate script that can be called for information, and / or as part of the upload script)
- Warn / Error in case user has included inconsistent / incomplete payload
- Let user know that the upload worked by requesting the data after upload again
- ... your ideas?

## 1.6 More examples

- Check how we handle CLI-based operations e.g. in `SN_reservation.py` or `upload_FlexTail_measurement.py`
- See how the interaction with the DB is done from inside a GUI in `main.py`
- Learn how to upload from external software like FADAPro (ATLAS internal - see `analysis/DB_interface.py` in FADAPro project)